In [0]:
%pip install lightgbm==4.6.0

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow

model_uri = "models:/nyctaxi_dev.ml.nyctaxi_trip_duration_lgbm_model/1"
loaded_model = mlflow.pyfunc.load_model(model_uri)

print("Model loaded successfully")

In [0]:
import mlflow
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

import pandas as pd

FEATURE_SCHEMA = {
    "vendor_name": "string",
    "passenger_count": "Int32",
    "trip_distance": "float64",
    "rate_code": "string",
    "payment_type": "string",
    "fare_amount": "float64",
    "tip_amount": "float64",
    "total_amount": "float64",
    "surcharge": "float64",
    "mta_tax": "float64",
    "tolls_amount": "float64",
    "start_lon": "float64",
    "start_lat": "float64",
    "end_lon": "float64",
    "end_lat": "float64",
    "pickup_hour": "Int32",
    "pickup_day_of_week": "Int32",
    "pickup_month": "Int32",
    "pickup_is_weekend": "Int32",
    "avg_speed_mph": "float64",
    "label": "float64",
    "is_rush_hour": "Int32",
    "is_night": "Int32",
    "haversine_distance": "float64",
    "manhattan_distance": "float64",
    "fare_per_mile": "float64",
    "tip_ratio": "float64",
    "tolls_ratio": "float64",
    "predicted_trip_duration": "float64"
}

def apply_feature_schema(df: pd.DataFrame, schema_map: dict = FEATURE_SCHEMA) -> pd.DataFrame:
    df = df.copy()

    for col, dtype in schema_map.items():
        if col not in df.columns:
            continue

        if dtype == "string":
            df[col] = df[col].apply(lambda x: "unknown" if pd.isna(x) else str(x)).astype("string")

        elif dtype == "Int32":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int32")

        elif dtype == "float64":
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    return df

try:
    spark
    print("Spark session already exists.")
except NameError:
    print("Spark session not found. Creating a new one...")
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

print("Spark version:", spark.version)

model_uri = "models:/nyctaxi_dev.ml.nyctaxi_trip_duration_lgbm_model@champion"
loaded_model = mlflow.pyfunc.load_model(model_uri)

# 读取原始打分数据
scoring_df = (
    spark.table("nyctaxi_dev.gold.gld_taxi_training_features")
    .limit(100000)
    .toPandas()
)
scoring_df = apply_feature_schema(scoring_df)

print("Input dtypes:")
print(scoring_df.dtypes)

# 1. 重新计算训练时使用的特征
scoring_df["is_rush_hour"] = scoring_df["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
scoring_df["is_night"] = ((scoring_df["pickup_hour"] < 6) | (scoring_df["pickup_hour"] >= 22)).astype(int)

def haversine_np(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(
        np.radians,
        [lon1, lat1, lon2, lat2]
    )
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km

scoring_df["haversine_distance"] = haversine_np(
    scoring_df["start_lon"],
    scoring_df["start_lat"],
    scoring_df["end_lon"],
    scoring_df["end_lat"]
)

scoring_df["manhattan_distance"] = (
    haversine_np(scoring_df["start_lon"], scoring_df["start_lat"], scoring_df["end_lon"], scoring_df["start_lat"]) +
    haversine_np(scoring_df["start_lon"], scoring_df["start_lat"], scoring_df["start_lon"], scoring_df["end_lat"])
)

scoring_df["fare_per_mile"] = np.where(
    scoring_df["trip_distance"] > 0,
    scoring_df["fare_amount"] / scoring_df["trip_distance"],
    np.nan
)

scoring_df["tip_ratio"] = np.where(
    scoring_df["fare_amount"] > 0,
    scoring_df["tip_amount"] / scoring_df["fare_amount"],
    np.nan
)

scoring_df["tolls_ratio"] = np.where(
    scoring_df["total_amount"] > 0,
    scoring_df["tolls_amount"] / scoring_df["total_amount"],
    np.nan
)

# 2. 只保留模型需要的输入列，顺序也和训练保持一致
model_input_cols = [
    "vendor_name",
    "passenger_count",
    "trip_distance",
    "rate_code",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "surcharge",
    "mta_tax",
    "tolls_amount",
    "start_lon",
    "start_lat",
    "end_lon",
    "end_lat",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "pickup_is_weekend",
    "is_rush_hour",
    "is_night",
    "haversine_distance",
    "manhattan_distance",
    "fare_per_mile",
    "tip_ratio",
    "tolls_ratio"
]

model_input_df = scoring_df[model_input_cols].copy()

# 3. 预测
preds = loaded_model.predict(model_input_df)

# 4. 写回结果
scoring_df["predicted_trip_duration"] = preds

pred_df = spark.createDataFrame(scoring_df)

(pred_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("nyctaxi_dev.ml.pred_trip_duration_batch"))

print("Batch inference completed.")

In [0]:
spark.table("nyctaxi_dev.bronze.yellow_taxi_raw").printSchema()